In [2]:
# @title Калькулятор гаражів: Графіки + Вибір поверховості
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import math
import matplotlib.pyplot as plt
import numpy as np

# --- Нормативні дані ---
data_norms = {
    "Поверховість": ["1", "2", "3", "4", "5", "6", "7-9"],
    "Норма забудови (м²/місце)": [25, 15, 10, 8, 6, 5, 4],
    "Норма ділянки (м²/місце)": [30, 20, 14, 12, 10, 8, 6]
}
norms_df = pd.DataFrame(data_norms)

# --- Логіка розрахунків (Табличні, Інтерполяція, -25%) ---
def get_tab_val(n, vals):
    if n <= 10: return vals[0]
    if n <= 50: return vals[1]
    if n <= 100: return vals[2]
    if n <= 300: return vals[3]
    return vals[4]

def interpolate(n, points, values):
    if n <= points[0]: return values[0]
    if n >= points[-1]: return values[-1]
    for i in range(len(points) - 1):
        if points[i] < n <= points[i+1]:
            if isinstance(values[i], str) or isinstance(values[i+1], str):
                return "погодж."
            res = values[i] + (n - points[i]) * (values[i+1] - values[i]) / (points[i+1] - points[i])
            return round(res, 1)
    return values[-1]

def reduce_25(val):
    if isinstance(val, (int, float)):
        return round(val * 0.75, 1)
    return val

def get_full_row(n):
    p = [10, 50, 100, 300, 1000]
    v_res = [10, 15, 25, 35, 50]
    v_ends = [10, 10, 15, 25, 35]
    v_pub = [10, 10, 15, 25, 25]
    v_sch = [15, 25, 25, 50, "погодж."]
    v_hosp = [25, 50, "погодж.", "погодж.", "погодж."]

    # Розрахунок по категоріях
    t_res = get_tab_val(n, v_res); i_res = interpolate(n, p, v_res); res_25 = reduce_25(t_res)
    t_ends = get_tab_val(n, v_ends); i_ends = interpolate(n, p, v_ends); ends_25 = reduce_25(t_ends)
    t_pub = get_tab_val(n, v_pub); i_pub = interpolate(n, p, v_pub); pub_25 = reduce_25(t_pub)
    t_sch = get_tab_val(n, v_sch); i_sch = interpolate(n, p, v_sch); sch_25 = reduce_25(t_sch)
    t_hosp = get_tab_val(n, v_hosp); i_hosp = interpolate(n, p, v_hosp); hosp_25 = reduce_25(t_hosp)

    return [t_res, i_res, res_25, t_ends, i_ends, ends_25,
            t_pub, i_pub, pub_25, t_sch, i_sch, sch_25, t_hosp, i_hosp, hosp_25]

# --- UI Елементи ---
style_layout = widgets.Layout(width='98%')

# 1. Основні параметри
calc_mode = widgets.ToggleButtons(
    options=[('К-сть машиномісць', 'places'), ('Площа забудови (м²)', 'build_area'), ('Площа ділянки (м²)', 'total_area')],
    description='Режим:', button_style='info', layout=style_layout
)
input_val = widgets.FloatText(value=100, description='Значення:', layout=widgets.Layout(width='200px'))

# 2. Вибір поверховості
floor_select = widgets.SelectMultiple(
    options=data_norms["Поверховість"],
    value=['1'], # Дефолт - 1 поверх
    description='Поверхи:',
    rows=7,
    layout=widgets.Layout(width='150px')
)

btn_all = widgets.Button(description="Всі поверхи", button_style='warning', icon='list', layout=widgets.Layout(width='120px'))

def select_all_floors(b):
    floor_select.value = tuple(data_norms["Поверховість"])
btn_all.on_click(select_all_floors)

output_panel = widgets.Output()

# --- Логіка побудови графіків ---
def plot_charts(df):
    # Підготовка даних
    floors = df["Поверховість"].astype(str)
    y_pos = np.arange(len(floors))

    # --- Графік 1: Геометрія (Subplots для різних масштабів) ---
    fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # Зліва: Машиномісця
    ax1.barh(y_pos, df["М/місця (ціле)"], color='#4caf50', alpha=0.7, label='М/місця')
    ax1.set_xlabel('Кількість')
    ax1.set_title('Місткість (шт.)')
    ax1.grid(axis='x', linestyle='--', alpha=0.5)
    for i, v in enumerate(df["М/місця (ціле)"]):
        ax1.text(v, i, f" {v}", va='center')

    # Справа: Площі
    width = 0.35
    ax2.barh(y_pos - width/2, df["Пл. забудови (м²)"], height=width, color='#2196f3', label='Пл. забудови')
    ax2.barh(y_pos + width/2, df["Пл. ділянки (м²)"], height=width, color='#ff9800', label='Пл. ділянки')
    ax2.set_xlabel('Площа, м²')
    ax2.set_title('Використання землі')
    ax2.legend()
    ax2.grid(axis='x', linestyle='--', alpha=0.5)

    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(floors)
    ax1.set_ylabel("Поверховість")
    fig1.suptitle('Порівняння ефективності по поверхах', fontsize=14)
    plt.tight_layout()
    plt.show()

    # --- Графік 2: Санітарні розриви (Для першого обраного варіанту) ---
    # Беремо перший рядок з відфільтрованої таблиці як приклад
    if len(df) > 0:
        row = df.iloc[0]
        floor_label = row["Поверховість"]
        cars = row["М/місця (ціле)"]

        objects = ['Житло', 'Торці', 'Громадські', 'Школи', 'Лікарні']

        # Дані для графіку (фільтруємо "погодж." для відображення)
        raw_interp = [row["Житло (інтерп.)"], row["Торці (інтерп.)"], row["Гром. (інтерп.)"], row["Школи (інтерп.)"], row["Лікарні (інтерп.)"]]
        raw_25 = [row["Житло (-25%)"], row["Торці (-25%)"], row["Гром. (-25%)"], row["Школи (-25%)"], row["Лікарні (-25%)"]]

        # Заміна "погодж." на 0 для графіку (або max)
        vals_interp = [x if isinstance(x, (int, float)) else 0 for x in raw_interp]
        vals_25 = [x if isinstance(x, (int, float)) else 0 for x in raw_25]

        fig2, ax3 = plt.subplots(figsize=(10, 4))
        x = np.arange(len(objects))
        width = 0.35

        rects1 = ax3.bar(x - width/2, vals_interp, width, label='Інтерполяція', color='#5c6bc0')
        rects2 = ax3.bar(x + width/2, vals_25, width, label='Пільга (-25% від табл.)', color='#66bb6a')

        ax3.set_ylabel('Відстань, м')
        ax3.set_title(f'Санітарні обмеження (Поверх: {floor_label}, Авто: {cars})')
        ax3.set_xticks(x)
        ax3.set_xticklabels(objects)
        ax3.legend()
        ax3.grid(axis='y', linestyle='--', alpha=0.3)

        # Підписи значень
        ax3.bar_label(rects1, padding=3, fmt='%.1f')
        ax3.bar_label(rects2, padding=3, fmt='%.1f')

        plt.tight_layout()
        plt.show()

# --- Головна функція оновлення ---
def update_table(*args):
    with output_panel:
        output_panel.clear_output()
        val, mode = input_val.value, calc_mode.value
        selected_floors = list(floor_select.value)

        if val <= 0 or not selected_floors:
            if not selected_floors: display(HTML("<b style='color:red'>Оберіть хоча б один поверх!</b>"))
            return

        # 1. Розрахунок повної таблиці
        full_res = norms_df.copy()

        if mode == 'places':
            full_res["М/місця (точне)"] = val; full_res["М/місця (ціле)"] = int(val)
        elif mode == 'build_area':
            full_res["М/місця (точне)"] = val / full_res["Норма забудови (м²/місце)"]; full_res["М/місця (ціле)"] = full_res["М/місця (точне)"].apply(math.floor)
        else:
            full_res["М/місця (точне)"] = val / full_res["Норма ділянки (м²/місце)"]; full_res["М/місця (ціле)"] = full_res["М/місця (точне)"].apply(math.floor)

        full_res["Пл. забудови (м²)"] = full_res["М/місця (точне)"] * full_res["Норма забудови (м²/місце)"]
        full_res["Пл. ділянки (м²)"] = full_res["М/місця (точне)"] * full_res["Норма ділянки (м²/місце)"]
        full_res["Факт. пл. забуд. на 1 м/м"] = full_res.apply(lambda x: x["Пл. забудови (м²)"] / x["М/місця (ціле)"] if x["М/місця (ціле)"] > 0 else 0, axis=1)
        full_res["Факт. пл. ділянки на 1 м/м"] = full_res.apply(lambda x: x["Пл. ділянки (м²)"] / x["М/місця (ціле)"] if x["М/місця (ціле)"] > 0 else 0, axis=1)

        dist_cols = ["Житло (табл.)", "Житло (інтерп.)", "Житло (-25%)", "Торці (табл.)", "Торці (інтерп.)", "Торці (-25%)", "Гром. (табл.)", "Гром. (інтерп.)", "Гром. (-25%)", "Школи (табл.)", "Школи (інтерп.)", "Школи (-25%)", "Лікарні (табл.)", "Лікарні (інтерп.)", "Лікарні (-25%)"]
        dist_data = full_res["М/місця (ціле)"].apply(get_full_row).tolist()
        full_res = pd.concat([full_res, pd.DataFrame(dist_data, columns=dist_cols)], axis=1)

        # 2. Фільтрація по вибраних поверхах
        res = full_res[full_res["Поверховість"].isin(selected_floors)].copy()

        # Порядок
        order = ["Поверховість", "Норма забудови (м²/місце)", "Факт. пл. забуд. на 1 м/м", "Норма ділянки (м²/місце)", "Факт. пл. ділянки на 1 м/м",
                 "М/місця (точне)", "М/місця (ціле)", "Пл. забудови (м²)", "Пл. ділянки (м²)",
                 "Житло (табл.)", "Житло (інтерп.)", "Житло (-25%)", "Торці (табл.)", "Торці (інтерп.)", "Торці (-25%)", "Гром. (табл.)", "Гром. (інтерп.)", "Гром. (-25%)", "Школи (табл.)", "Школи (інтерп.)", "Школи (-25%)", "Лікарні (табл.)", "Лікарні (інтерп.)", "Лікарні (-25%)"]
        res = res[order]

        # 3. Відображення таблиці
        input_idx = 6 if mode == 'places' else (7 if mode == 'build_area' else 8)
        styles = [{'selector': 'td', 'props': [('font-size', '14px'), ('text-align', 'center'), ('padding', '8px')]}]

        for i, col in enumerate(res.columns):
            bg = "#f9f9f9"
            if i == input_idx: bg = "#bbdefb"
            elif i in [6, 7, 8]: bg = "#c8e6c9"
            elif i in [9, 10, 11]: bg = "#fff9c4"
            elif i in [12, 13, 14]: bg = "#ffe0b2"
            elif i in [15, 16, 17]: bg = "#e1bee7"
            elif i in [18, 19, 20]: bg = "#b2dfdb"
            elif i in [21, 22, 23]: bg = "#ffcdd2"
            styles.append({'selector': f'th.col{i}', 'props': [('background-color', bg), ('font-size', '14px'), ('max-width', '75px'), ('height', '80px'), ('border', '1px solid #ccc'), ('padding', '5px')]})
        styles.append({'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]})

        display(HTML(f"<h3 style='margin-bottom:5px;'>Розрахунок для: {val:g}</h3>"))
        display(res.style.set_table_styles(styles).hide(axis='index').format(precision=1, na_rep="-"))

        # Примітка 2
        if res["М/місця (ціле)"].between(101, 300).any():
            display(HTML("<div style='background-color:#fff3cd;padding:10px;margin-top:10px;'>⚠️ <b>Примітка 2:</b> Для секційних житлових будинків (101-300 авто) відстань ≥ 50 м.</div>"))

        # 4. Відображення графіків
        display(HTML("<hr><h4>📊 Візуалізація результатів</h4>"))
        plot_charts(res)

# Компонування UI
left_box = widgets.VBox([calc_mode, input_val])
right_box = widgets.HBox([
    widgets.VBox([widgets.Label("Вибір поверхів:"), floor_select]),
    widgets.VBox([widgets.Label(" "), btn_all]) # Кнопка з відступом
])
ui = widgets.HBox([left_box, right_box], layout=widgets.Layout(justify_content='space-between'))

calc_mode.observe(update_table, names='value')
input_val.observe(update_table, names='value')
floor_select.observe(update_table, names='value')

display(ui, output_panel)
update_table()

Output()

In [1]:
# @title Калькулятор: Повний склад (+ Торці табличні)
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import math

# --- Нормативні дані ---
data_norms = {
    "Поверховість": ["1", "2", "3", "4", "5", "6", "7-9"],
    "Норма забудови (м²/місце)": [25, 15, 10, 8, 6, 5, 4],
    "Норма ділянки (м²/місце)": [30, 20, 14, 12, 10, 8, 6]
}
norms_df = pd.DataFrame(data_norms)

# --- Логіка табличних значень ---
def get_tab_val(n, vals):
    if n <= 10: return vals[0]
    if n <= 50: return vals[1]
    if n <= 100: return vals[2]
    if n <= 300: return vals[3]
    return vals[4]

# --- Логіка інтерполяції ---
def interpolate(n, points, values):
    if n <= points[0]: return values[0]
    if n >= points[-1]: return values[-1]
    for i in range(len(points) - 1):
        if points[i] < n <= points[i+1]:
            if isinstance(values[i], str) or isinstance(values[i+1], str):
                return "погодж."
            res = values[i] + (n - points[i]) * (values[i+1] - values[i]) / (points[i+1] - points[i])
            return round(res, 1)
    return values[-1]

# Функція для обчислення -25%
def reduce_25(val):
    if isinstance(val, (int, float)):
        return round(val * 0.75, 1)
    return val

def get_full_row(n):
    p = [10, 50, 100, 300, 1000]

    # Базові масиви значень
    v_res = [10, 15, 25, 35, 50]       # Житло
    v_ends = [10, 10, 15, 25, 35]      # Торці
    v_pub = [10, 10, 15, 25, 25]       # Громадські
    v_sch = [15, 25, 25, 50, "погодж."] # Школи
    v_hosp = [25, 50, "погодж.", "погодж.", "погодж."] # Лікарні

    # --- 1. Житло ---
    t_res = get_tab_val(n, v_res)
    i_res = interpolate(n, p, v_res)
    res_25 = reduce_25(t_res) # -25% від табличного

    # --- 2. Торці ---
    t_ends = get_tab_val(n, v_ends)     # Табличне
    i_ends = interpolate(n, p, v_ends)  # Інтерполяція
    ends_25 = reduce_25(t_ends)         # -25% від табличного

    # --- 3. Громадські ---
    t_pub = get_tab_val(n, v_pub)
    i_pub = interpolate(n, p, v_pub)
    pub_25 = reduce_25(t_pub)

    # --- 4. Школи ---
    t_sch = get_tab_val(n, v_sch)
    i_sch = interpolate(n, p, v_sch)
    sch_25 = reduce_25(t_sch)

    # --- 5. Лікарні ---
    t_hosp = get_tab_val(n, v_hosp)
    i_hosp = interpolate(n, p, v_hosp)
    hosp_25 = reduce_25(t_hosp)

    return [t_res, i_res, res_25,
            t_ends, i_ends, ends_25, # Додано t_ends
            t_pub, i_pub, pub_25,
            t_sch, i_sch, sch_25,
            t_hosp, i_hosp, hosp_25]

# --- UI ---
calc_mode = widgets.ToggleButtons(
    options=[('К-сть машиномісць', 'places'), ('Площа забудови (м²)', 'build_area'), ('Площа ділянки (м²)', 'total_area')],
    description='Режим:', button_style='info'
)
input_val = widgets.FloatText(value=100, description='Значення:', layout=widgets.Layout(width='200px'))
output_panel = widgets.Output()

def update_table(*args):
    with output_panel:
        output_panel.clear_output()
        val, mode = input_val.value, calc_mode.value
        if val <= 0: return
        res = norms_df.copy()

        # Розрахунок місць
        if mode == 'places':
            res["М/місця (точне)"] = val; res["М/місця (ціле)"] = int(val)
        elif mode == 'build_area':
            res["М/місця (точне)"] = val / res["Норма забудови (м²/місце)"]; res["М/місця (ціле)"] = res["М/місця (точне)"].apply(math.floor)
        else:
            res["М/місця (точне)"] = val / res["Норма ділянки (м²/місце)"]; res["М/місця (ціле)"] = res["М/місця (точне)"].apply(math.floor)

        # Геометрія
        res["Пл. забудови (м²)"] = res["М/місця (точне)"] * res["Норма забудови (м²/місце)"]
        res["Пл. ділянки (м²)"] = res["М/місця (точне)"] * res["Норма ділянки (м²/місце)"]
        res["Факт. пл. забуд. на 1 м/м"] = res.apply(lambda x: x["Пл. забудови (м²)"] / x["М/місця (ціле)"] if x["М/місця (ціле)"] > 0 else 0, axis=1)
        res["Факт. пл. ділянки на 1 м/м"] = res.apply(lambda x: x["Пл. ділянки (м²)"] / x["М/місця (ціле)"] if x["М/місця (ціле)"] > 0 else 0, axis=1)

        # Санітарні розриви
        dist_cols = [
            "Житло (табл.)", "Житло (інтерп.)", "Житло (-25%)",
            "Торці (табл.)", "Торці (інтерп.)", "Торці (-25%)",
            "Гром. (табл.)", "Гром. (інтерп.)", "Гром. (-25%)",
            "Школи (табл.)", "Школи (інтерп.)", "Школи (-25%)",
            "Лікарні (табл.)", "Лікарні (інтерп.)", "Лікарні (-25%)"
        ]
        dist_data = res["М/місця (ціле)"].apply(get_full_row).tolist()
        res = pd.concat([res, pd.DataFrame(dist_data, columns=dist_cols)], axis=1)

        # Порядок стовпців
        order = [
            "Поверховість", "Норма забудови (м²/місце)", "Факт. пл. забуд. на 1 м/м",
            "Норма ділянки (м²/місце)", "Факт. пл. ділянки на 1 м/м",
            "М/місця (точне)", "М/місця (ціле)", "Пл. забудови (м²)", "Пл. ділянки (м²)",
            # Санітарія
            "Житло (табл.)", "Житло (інтерп.)", "Житло (-25%)",
            "Торці (табл.)", "Торці (інтерп.)", "Торці (-25%)",
            "Гром. (табл.)", "Гром. (інтерп.)", "Гром. (-25%)",
            "Школи (табл.)", "Школи (інтерп.)", "Школи (-25%)",
            "Лікарні (табл.)", "Лікарні (інтерп.)", "Лікарні (-25%)"
        ]
        res = res[order]

        # Стилізація
        input_idx = 6 if mode == 'places' else (7 if mode == 'build_area' else 8)

        styles = [
            {'selector': 'td', 'props': [('font-size', '14px'), ('text-align', 'center'), ('padding', '8px')]}
        ]

        for i, col in enumerate(res.columns):
            bg = "#f9f9f9"

            # --- Геометрія ---
            if i == input_idx: bg = "#bbdefb"
            elif i in [6, 7, 8]: bg = "#c8e6c9"

            # --- Санітарія (Кольорові зони) ---
            elif i in [9, 10, 11]: bg = "#fff9c4"    # Жовтий (Житло)
            elif i in [12, 13, 14]: bg = "#ffe0b2"   # Помаранчевий (Торці)
            elif i in [15, 16, 17]: bg = "#e1bee7"   # Бузковий (Громадські)
            elif i in [18, 19, 20]: bg = "#b2dfdb"   # Блакитний (Школи)
            elif i in [21, 22, 23]: bg = "#ffcdd2"   # Рожевий (Лікарні)

            styles.append({'selector': f'th.col{i}', 'props': [
                ('background-color', bg), ('font-size', '14px'), ('font-weight', 'bold'),
                ('max-width', '75px'), ('height', '80px'), ('border', '1px solid #ccc'),
                ('white-space', 'normal'), ('word-wrap', 'break-word'), ('padding', '5px'),
                ('color', 'black')
            ]})

        styles.append({'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]})

        display(HTML(f"<h3 style='margin-bottom:10px;'>Результати для: {val:g}</h3>"))
        display(res.style.set_table_styles(styles).hide(axis='index').format(precision=1, na_rep="-"))

        # --- Попередження для секційних (Примітка 2) ---
        check_sectional = res["М/місця (ціле)"].between(101, 300).any()
        if check_sectional:
            display(HTML("""
            <div style='background-color: #fff3cd; color: #856404; padding: 15px; border: 1px solid #ffeeba; border-radius: 5px; margin-top: 15px;'>
                <strong>⚠️ Увага (Примітка 2):</strong><br>
                Виявлено кількість машиномісць у діапазоні <b>101–300</b>.<br>
                Відстань від <b>секційних житлових будинків</b> до відкритих майданчиків, розміщених уздовж поздовжніх фасадів, слід приймати <b>не менше 50 м</b>.
            </div>
            """))

calc_mode.observe(update_table, names='value'); input_val.observe(update_table, names='value')
display(widgets.VBox([calc_mode, input_val, output_panel])); update_table()